In [1]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 100)


font = {'size'   : 16}
matplotlib.rc('font', **font)
matplotlib.rc('xtick', labelsize=14) 
matplotlib.rc('ytick', labelsize=14) 
matplotlib.rcParams.update({'figure.autolayout': False})
matplotlib.rcParams['figure.dpi'] = 300

In [2]:
PokemonSet = pd.read_csv('pokemon.csv')
target = PokemonSet['is_legendary']

### Importante

Para que el algoritmo realice un mejor trabajo, filtramos los features que tiene el DataSet. En este caso eliminamos los siguientes features que no aportan información relevante (según criterio propio):
- name
- japanese_name
- pokedex_number
- classification
- against_? 

Además, debido a que abilities es un string que contiene una o más habilidades por pokemones, es una feature un poco más difícil de trabajar, por lo que optamos por eliminarla del DataSet:
- abilities

También eliminamos el feature que nos dice directamente si el pokemon es legendario (pero se utilizará para validar los resultados, por eso se guardo la variable 'y' antes):
- is_legendary

Eliminamos los siguientes 4 features debido a que en el trabajo anterior se determinó que estas están directamente relacionadas con los pokemones legendarios (concentran a los legendarios dentro de unos rangos pequeños), evitando así hacer "trampa" con el modelo y determinar si es que se puede predecir correctamente utilizando las demás features:
- capture_rate
- percentage_male
- base_egg_steps 
- experience_growth

Por último, eliminamos la siguiente feature debido a que es la suma de otras features, por lo que la vuelve redundante para el modelo:
- base_total

In [3]:
PokemonSet = PokemonSet.drop(['name', 
                            'japanese_name', 
                            'pokedex_number',
                            'classfication',
                            'against_fire',
                            'against_water',
                            'against_electric',
                            'against_grass',
                            'against_ice',
                            'against_fight',
                            'against_poison',
                            'against_ground',
                            'against_flying',
                            'against_psychic',
                            'against_bug',
                            'against_rock',
                            'against_ghost',
                            'against_dragon',
                            'against_dark',
                            'against_steel',
                            'against_fairy',
                            'against_normal',
                            'is_legendary',
                            'capture_rate',
                            'percentage_male',
                            'base_egg_steps',
                            'abilities',
                            'experience_growth',
                            'base_total'],
                            axis=1)

PokemonSet

,attack,base_happiness,defense,height_m,hp,sp_attack,sp_defense,speed,type1,type2,weight_kg,generation
0,49,70,49,0.7,45,65,65,45,grass,poison,6.9,1
1,62,70,63,1.0,60,80,80,60,grass,poison,13.0,1
2,100,70,123,2.0,80,122,120,80,grass,poison,100.0,1
3,52,70,43,0.6,39,60,50,65,fire,NaN,8.5,1
4,64,70,58,1.1,58,80,65,80,fire,NaN,19.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...
796,101,0,103,9.2,97,107,101,61,steel,flying,999.9,7
797,181,0,131,0.3,59,59,31,109,grass,steel,0.1,7
798,101,0,53,5.5,223,97,53,43,dark,dragon,888.0,7
799,107,0,101,2.4,97,127,89,79,psychic,NaN,230.0,7


Las variables que corresponden a características cualitativas deben de convertirse en valores numéricos para poder ser analizados, así que convertimos las variables 'type1' y 'type2' en una matriz que represente con '1' al tipo de pokemon que si corresponde, y '0' al tipo que no (de esta forma eliminamos los NaN's de la tabla de datos).

In [4]:
tipos = (pd.get_dummies(PokemonSet['type1'], dtype=int)
           .add(pd.get_dummies(PokemonSet['type2'], dtype=int), fill_value=0)
           .astype(int)
           .add_prefix('tipo_'))

PokemonSet = pd.concat([PokemonSet.drop(columns=['type1', 'type2']), tipos], axis=1)

In [5]:
PokemonSet

,attack,base_happiness,defense,height_m,hp,sp_attack,sp_defense,speed,weight_kg,generation,tipo_bug,tipo_dark,tipo_dragon,tipo_electric,tipo_fairy,tipo_fighting,tipo_fire,tipo_flying,tipo_ghost,tipo_grass,tipo_ground,tipo_ice,tipo_normal,tipo_poison,tipo_psychic,tipo_rock,tipo_steel,tipo_water
0,49,70,49,0.7,45,65,65,45,6.9,1,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0
1,62,70,63,1.0,60,80,80,60,13.0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0
2,100,70,123,2.0,80,122,120,80,100.0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0
3,52,70,43,0.6,39,60,50,65,8.5,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
4,64,70,58,1.1,58,80,65,80,19.0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
796,101,0,103,9.2,97,107,101,61,999.9,7,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0
797,181,0,131,0.3,59,59,31,109,0.1,7,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0
798,101,0,53,5.5,223,97,53,43,888.0,7,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
799,107,0,101,2.4,97,127,89,79,230.0,7,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0


En el caso de los NaN's que se obtienen en la altura y peso, los dejaremos tal cual como están, pues aunque ya sabemos del EDA que este es un error del DataSet debido a que mezcla las características de un pokemon original con alguna variación suya, no queremos modificar mucho los datos que recibimos. De esta manera nos ajustamos mejor a un caso realista en el que podría haber también un error en los datos recolectados.

In [6]:
from sklearn import metrics
from sklearn.model_selection import cross_validate, StratifiedKFold, cross_val_predict, GridSearchCV
from sklearn.ensemble import RandomForestClassifier

In [7]:
PokemonSet.shape

(801, 28)

In [8]:
features = PokemonSet

In [9]:
model = RandomForestClassifier(random_state=42)

model.get_params()

{'bootstrap': True,
 'ccp_alpha': 0.0,
 'class_weight': None,
 'criterion': 'gini',
 'max_depth': None,
 'max_features': 'sqrt',
 'max_leaf_nodes': None,
 'max_samples': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'monotonic_cst': None,
 'n_estimators': 100,
 'n_jobs': None,
 'oob_score': False,
 'random_state': 42,
 'verbose': 0,
 'warm_start': False}

In [10]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=101)

cv_inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)
cv_outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=2)

In [11]:
scoring = {'f1': 'f1',
           'precision': 'precision',
           'recall': 'recall'}

In [12]:
scores = cross_validate(
    estimator=model,
    X=features,
    y=target,
    cv=cv,
    scoring=scoring,
    return_train_score=True)

scores

{'fit_time': array([0.71166158, 0.75501442, 0.53865504, 0.59589219, 0.5098443 ]),
 'score_time': array([0.05901909, 0.06861997, 0.04325151, 0.06619   , 0.07958436]),
 'test_f1': array([0.57142857, 0.69565217, 0.88      , 0.52631579, 0.6       ]),
 'train_f1': array([1., 1., 1., 1., 1.]),
 'test_precision': array([0.85714286, 0.88888889, 1.        , 1.        , 1.        ]),
 'train_precision': array([1., 1., 1., 1., 1.]),
 'test_recall': array([0.42857143, 0.57142857, 0.78571429, 0.35714286, 0.42857143]),
 'train_recall': array([1., 1., 1., 1., 1.])}

In [13]:
print(f"recall test score: {scores['test_recall'].mean():.4f} ± {scores['test_recall'].std():.4f}")
print(f"recall train score: {scores['train_recall'].mean():.4f} ± {scores['train_recall'].std():.4f}")

recall test score: 0.5143 ± 0.1525
recall train score: 1.0000 ± 0.0000


In [14]:
parameters = {'max_features':['sqrt', 4, None],
              'n_estimators':[100, 200, 500],
              'min_samples_split': [2, 5, 10], 
              'min_samples_leaf': [1, 2, 4],
              'class_weight': [None, 'balanced_subsample']}

In [ ]:
grid = GridSearchCV(
    estimator=model,
    param_grid=parameters,
    cv=cv_inner,
    scoring=scoring,
    refit='recall',
    return_train_score=True,
    n_jobs=-1,
    verbose=1
)

grid.fit(features, target)

Fitting 5 folds for each of 162 candidates, totalling 810 fits


In [ ]:
print(grid.best_params_)

In [ ]:
scoresGrid = pd.DataFrame(grid.cv_results_)
scoresCV = (scoresGrid[['params', 'mean_test_f1', 'std_test_f1',
                        'mean_test_recall', 'mean_test_precision', 'mean_train_f1']]
            .sort_values(by='mean_test_f1', ascending=False))

In [ ]:
scoresCV

### Evaluación con validación cruzada anidada

Los resultados de la búsqueda de hiperparámetros anterior (`best_params_`, `scoresCV`) son algo optimistas, porque los hiperparámetros se eligieron con los mismos folds con los que se miden. Para estimar el rendimiento de forma justa, se ejecuta la búsqueda completa (`grid`, con `cv_inner`) dentro de una segunda validación cruzada (`cv_outer`): en cada fold externo, el grid elige los hiperparámetros sin ver los datos de ese fold, y luego se evalúa ahí. Los valores de esta sección son los que se reportan como rendimiento esperado del modelo.

In [ ]:
nested = cross_validate(
    estimator=grid,
    X=features,
    y=target,
    cv=cv_outer,
    scoring=scoring
)

for m in scoring:
    print(f"{m} test score (anidado): {nested[f'test_{m}'].mean():.4f} ± {nested[f'test_{m}'].std():.4f}")

### Modelo final

La validación anidada solo estima el rendimiento. El modelo final es el que deja `grid.fit(features, target)`: los mejores hiperparámetros reentrenados con todos los datos.

In [ ]:
modelo_final = grid.best_estimator_

importancias = pd.Series(modelo_final.feature_importances_, index=features.columns).sort_values(ascending=False)
importancias.head(10)